In [1]:
import requests,json,random,string,base64,io
from datetime import datetime
from time import sleep

In [2]:
def file2b64(file_path): 
    with open(file_path, "rb") as file:
        return base64.b64encode(io.BytesIO(file.read()).getvalue()).decode("ascii")
def b642bin(val):return io.BytesIO(base64.b64decode(val.decode('utf-8') if isinstance(val,bytes) else val))

In [21]:
class PAClient:
    def __init__(self, url="https://4dtjs429x0.execute-api.us-east-1.amazonaws.com/"):self.url=url
    
    #user API
    def userCreate(self,name, balance,email=None):return self._post("user","create",{"name":name,"balance":balance,"email":email})
    def userGet(self, uid): return self._get("user",uid)
    def userDelete(self, uid): return self._delete("user",uid)
    def userCredit(self, uid,delta): return self._put("user",uid+"/credit",{"delta":delta})
    def userEnum(self,q='True'): return self._get("user", "enumerate",q={"query":q})
    def userSubscribe(self,uid,sub_dict): return self._put("user",uid+"/update",body={"subscription":sub_dict})
    
    #doc API
    def docCreate(self,content=None,owner=None,status="ready", parent=None):
        return self._post("doc","create",{'content':content,'owner':owner,'status':status,'parent':parent})
    def docUpdate(self, did, content=None, owner=None,status="ready",parent=None):
        return self._put("doc",did+'/update',body={'content':content,'owner':owner,'status':status,'parent':parent})
    def docFormat(self,did,fmt="json"):return self._get("doc",did+"/"+fmt)
    def docEnum(self,q='True'): return self._get("doc", "enumerate",q={"query":q})
    def docDelete(self,did):return self._delete("doc",did)
    def docMedia(self,mid): return self._getbin("doc","media/"+mid)
    def docPollForStatus(self,did,max_duration_ms=10000,poll_interval_ms=500,status="ready"):
        start,doc=datetime.now(),self.docFormat(did)
        while ((datetime.now()-start).total_seconds()*1000<max_duration_ms) and doc['status']!=status:
            sleep(poll_interval_ms*1.0/1000)
            doc=self.docFormat(did)
        return doc["status"]
            
    #transform API
    def transformCreate(self,name,cfg, owner, fees={},description=None, mapio=None):
        return self._post("transformer","create",{"name":name,"cfg":cfg,"owner":owner,"fees":fees, "description": description,"mapio":mapio})
    def transformGet(self,tid): return self._get("transformer",tid)
    def transformDelete(self,tid): return self._delete("transformer",tid)
    def transformUpdate(self,tid,name=None,cfg=None, owner=None, fees=None,description=None, mapio=None): 
        return self._put("transformer",tid+"/update",body={"name":name,"cfg":cfg,"owner":owner,"fees":fees, "description": description})
    def transformApply(self,tid, user, did_or_content):
        doc={"did": did_or_content} if isinstance(did_or_content,str) else {"content":did_or_content}
        return self._put("transformer",tid+"/apply",body={"owner":user,**doc})
    def transformEnum(self,q='True'): return self._get("transformer", "enumerate",q={"query":q})

    #graph api
    def feedAttach(self,name, description,owner,transformer,fees=None,source=None,nodetype=None,publishinfo=None):
        return self._post("graph","attach",{"name":name,"description":description,"owner":owner,"source":source,"transformer":transformer,"fees":fees,'nodetype':nodetype,'publishinfo':publishinfo})
    def feedUpdate(self, fid,name=None, description=None,owner=None,source=None,transformer=None,fees=None,publishinfo=None):
        return self._put("graph",fid+"/update",body={"name":name,"description":description,"owner":owner,"source":source,"transformer":transformer,"fees":fees,'publishinfo':publishinfo})
    def feedGet(self,fid): return self._get("graph",fid)
    def feedEnum(self,q='True'):return self._get("graph", "enumerate",q={"query":q})
    def feedDocs(self, fid, uid, since=datetime.min):return self._get("graph", fid+"/docs",q={"uid":uid,"since":since})
    def feedGenerate(self,fid, uid, since=datetime.utcnow(), maxnum=5, freq=0): return self._get("graph", fid+"/generate",q={"uid":uid,"since":since,"maxnum":maxnum,"freq":freq})
    def feedStatus(self, gid):return self._get("graph","status/"+gid)
    def feedPollForStatus(self,gid,max_duration_ms=10000,poll_interval_ms=500,status="ready"):
        start,task=datetime.now(),self.feedStatus(gid)
        while ((datetime.now()-start).total_seconds()*1000<max_duration_ms) and task['status']!=status:
            sleep(poll_interval_ms*1.0/1000)
            task=self.feedStatus(gid)
        return task["status"]
    def feedDelete(self,fid): return self._delete("graph",fid)
                                         
    def _post(self,resource,method,body):return json.loads(requests.post(self.url+"/"+resource+"/"+method, data=json.dumps(body)).content)
    def _get(self,resource,method,q=None): return json.loads(requests.get(self.url+"/"+resource+"/"+method,params=q).content)
    def _delete(self,resource,method):return json.loads(requests.delete(self.url+"/"+resource+"/"+method).content)
    def _put(self,resource,method,q=None,body=None):
        return json.loads(requests.put(self.url+"/"+resource+"/"+method,params=q,data=json.dumps(body) if body else None).content)
    def _getbin(self,resource,method): return requests.get(self.url+"/"+resource+"/"+method).content

In [22]:
feed_id = "8c23f9c4-dc7a-436b-a4d1-1f79a01d3c02"
user_id = "5e3db9b0-604b-44a1-8361-450f49693f77"
PAClient().feedGet(feed_id)
# PAClient().feedGenerate(feed_id, user_id, since=datetime.utcnow(), maxnum=1, freq=0)

{'message': 'Missing Authentication Token'}

In [24]:
str(datetime.utcnow())

'2023-07-24 21:43:56.086509'